# From RL to Bode
## Six-axis dynamics, impedance, a neural law — one framework

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/learn/intro/showcase_from_rl_to_bode.ipynb)

**The path.** Rigid-body robot dynamics, a classical impedance loop, a neural policy, and a Bode plot are capabilities that usually live in separate tools. Minilink puts them on one contract: every block is a `System`

$$
\dot x = f(x, u, t;\, p), \qquad y = h(x, u, t;\, p).
$$

A catalog plant, a controller, a neural net, and a wired diagram are the same kind of object. Simulate, animate, train, linearize, Bode — each tool takes that object. This notebook walks one closed loop from reinforcement learning to a frequency plot.

The plant is a six-axis UR5. Gravity-compensated joint impedance is the inner loop: its input is a joint set-point $r$, not a torque. A network $r = \pi_\theta(x)$ is the outer law; PPO trains $\theta$ so the tool holds $p^\star$ from random starts and payloads. After training the closed loop is still a `System`, so the same residual answers $P_z/F_z$ — the Bode of a learned controller — and a Lyapunov certificate says from how far it recovers.

**Contents**

1. [Six-axis dynamics](#1.-Six-axis-dynamics): the UR5 as a `System`
2. [The task](#2.-The-task)
3. [Impedance as the inner loop](#3.-Impedance-as-the-inner-loop)
4. [Before learning: Bode](#4.-Before-learning:-Bode)
5. [The cost](#5.-The-cost)
6. [A stochastic planning problem](#6.-A-stochastic-planning-problem)
7. [A neural net, trained with RL](#7.-A-neural-net,-trained-with-RL)
8. [The network is a controller block](#8.-The-network-is-a-controller-block)
9. [Watch the learned loop](#9.-Watch-the-learned-loop)
10. [After RL: Bode](#10.-After-RL:-Bode)
11. [How far can it be pushed and still recover?](#11.-How-far-can-it-be-pushed-and-still-recover?)
12. [Push the tool](#12.-Push-the-tool)
13. [One framework](#13.-One-framework)

The lab write-up with the same experiment: [ur5_impedance_rl](../teaching/ur5_impedance_rl.ipynb). Differentiable `f`: [showcase_jax](showcase_jax.ipynb). Library API: [11_reinforcement_learning](11_reinforcement_learning.ipynb).


In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

In [ ]:
import jax.numpy as jnp
import numpy as np

from minilink import CostFunction, DiagramSystem, JointImpedance, UR5Manipulator
from minilink.planning import (
    Gaussian,
    MonteCarloEvaluator,
    ReinforcementLearningPlanner,
    StochasticPlanningProblem,
    Uniform,
)

TRAINING_TIMESTEPS = 40_000  # about a minute on a laptop CPU
DT = 0.01  # control period of the set-point law (the impedance loop is continuous)
EPISODE = 3.0  # s

## 1. Six-axis dynamics

The first block is the physics: a six-degree-of-freedom UR5. Every minilink model is a `System` — one dynamics equation and one map per output port:

$$
\dot x = f(x, u, t;\, p), \qquad y = h(x, u, t;\, p).
$$

The object owns its ports (here the joint torque $u$ and a tool force $f$), its state $x$, and an initial condition $x_0$. Displaying it prints that block. `plot_diagram` draws the ports; `compute_trajectory` / `animate` run $f$ forward — no controller yet.

Joint coordinates $q\in\mathbb{R}^6$ and rates $\dot q$ make the state $x = (q,\dot q)$. The rigid-body equation of motion is

$$
H(q)\,\ddot q + C(q,\dot q)\,\dot q + g(q) = \tau.
$$

A world-frame force $f$ at the tool adds a generalized force through the Jacobian,

$$
\tau = u + J(q)^\top f,
$$

so the same plant later answers the frequency-domain question $p_z/f_z$. The catalog wrist has almost no inertia about its own axis: no damping works at a finite $\Delta t$. A $0.5\,\mathrm{kg}$ gripper on the last link fixes the *model*, not the integrator.

Below: print the block, draw it, then let the arm fall under gravity with $u = 0$ (open loop).


In [ ]:
class UR5WithToolForce(UR5Manipulator):
    def __init__(self):
        super().__init__()
        self.add_input_port("f", dim=3, nominal_value=np.zeros(3))

    def generalized_force(self, q, v, u, t=0.0, params=None):
        tau, f = u[:6], u[6:9]
        return tau + self.J(q, params).T @ f


arm = UR5WithToolForce()
arm.params["mass"] = np.asarray(arm.params["mass"], dtype=float) + np.array(
    [0, 0, 0, 0, 0, 0.5]
)
arm.params["inertia"] = np.asarray(arm.params["inertia"], dtype=float).copy()
arm.params["inertia"][5] += 0.02 * np.eye(3)

print(arm)
arm.plot_diagram()

# Open loop: zero torque, gravity only
arm.x0 = arm.q2x([0.0, -1.0, 1.2, -1.4, 0.0, 0.0], np.zeros(6))
traj_ol = arm.compute_trajectory(tf=3.0, dt=0.01, compile_backend="jax")
arm.plot_trajectory(traj_ol)
arm.animate(traj_ol, renderer="meshcat")

## 2. The task

Put the tool at a Cartesian point $p^\star$ and hold it there. Inverse kinematics gives the matching joint pose $q^\star$; the rest state is that pose at rest:

$$
q^\star = \mathrm{IK}(p^\star), \qquad x^\star = \begin{bmatrix} q^\star \\ 0 \end{bmatrix}.
$$

The rest of the notebook is how that same `System` reaches and holds $p^\star$ — first with impedance, then with a neural law, then with a Bode plot. Two things make a constant set-point $r \equiv q^\star$ imperfect: the arm may start away from $x^\star$, and the last link may carry an unknown payload that gravity compensation does not know. The **neural law will not output torques**. Impedance already maps a set-point to $\tau$. What is learned is an outer law

$$
r = \pi_\theta(x) \in \mathbb{R}^6
$$

— six joint set-points, given the current state — that drives the tool to $p^\star$ despite those draws.


In [ ]:
p_target = np.array([0.3, 0.4, 0.5])  # tool target [m]
q_target = arm.inverse_kinematics(p_target, q_guess=[0.0, -1.0, 1.2, -1.4, 0.0, 0.0])
x_target = arm.q2x(q_target, np.zeros(6))
print("tool target p* [m]:", p_target)
print("joint target q* [rad]:", np.round(q_target, 3))

## 3. Impedance as the inner loop

The second block is classical robot control, on the same contract as the arm. Gravity-compensated joint impedance is the law

$$
\tau = K_p\,(r - q) - K_d\,\dot q + g(q).
$$

Wrist gains are smaller because those joints are light. The operator `@` closes the loop: the plant the outer law will see is this diagram. Its input is the set-point $r$, not the torque $u$.

The boxes on $r$ and $x$ are the training domain — the set-points the outer law may command, and the states an episode may visit. Next we hold $r = q^\star$ and watch this loop alone, before any learned law.


In [ ]:
impedance = JointImpedance(
    arm,
    gravity_comp=True,
    Kp=[100.0, 100.0, 100.0, 10.0, 10.0, 10.0],
    Kd=[20.0, 20.0, 20.0, 1.0, 1.0, 1.0],
)
inner = impedance @ arm
inner.name = "Joint impedance loop"
inner.inputs["r"].lower_bound = q_target - 0.6
inner.inputs["r"].upper_bound = q_target + 0.6
inner.state.lower_bound = np.concatenate([q_target - 2.0, -10.0 * np.ones(6)])
inner.state.upper_bound = np.concatenate([q_target + 2.0, 10.0 * np.ones(6)])
inner.x0 = arm.q2x(q_target + np.array([0.4, -0.3, 0.3, -0.4, 0.3, 0.3]), np.zeros(6))
inner.inputs["r"].nominal_value = q_target
inner.plot_diagram()
print("inner-loop inputs:", list(inner.inputs), "| state dim:", inner.n)

## 4. Before learning: Bode

Hold the set-point at the joint target, $r(t) = q^\star$ — no neural law yet. Integrate from the offset start, plot the tool $p$, animate. Then linearize at $x^\star$ and Bode the load sensitivity — a frequency-domain question on the same inner-loop `System`:

$$
S_{p_z f_z}(j\omega) = \frac{P_z(j\omega)}{F_z(j\omega)}.
$$

This is the **before** picture: how much the tool moves when you push it, with a constant set-point.


In [ ]:
traj_imp = inner.compute_trajectory(tf=EPISODE, dt=0.005, compile_backend="jax")
inner.plot_trajectory(traj_imp, signals=((arm, "p"),))
inner.animate(traj_imp, renderer="meshcat")
print("tool at the end [m]:", np.round(arm.forward_kinematics(traj_imp.x[:6, -1]), 3))
print("target         [m]:", p_target)

# Same blocks, f exposed, for the before Bode (inner stays r-only so RL can train)
inner_f = DiagramSystem()
inner_f.name = "Impedance loop, tool force in"
inner_f.add_subsystem(impedance, "ctl")
inner_f.add_subsystem(arm, "arm")
inner_f.connect("arm", "y", "ctl", "y")
inner_f.connect("ctl", "u", "arm", "u")
inner_f.add_input_port("r", dim=6, nominal_value=q_target)
inner_f.connect("input", "r", "ctl", "r")
inner_f.add_input_port("f", dim=3, nominal_value=np.zeros(3))
inner_f.connect("input", "f", "arm", "f")
inner_f.plot_bode(x_target, of=("arm:p", 2), wrt=("f", 2), margins=False)

## 5. The cost

The planner minimizes the infinite-horizon cost $J = \int_0^\infty g(x,r,t)\,dt$ (no terminal $h$). The running cost is the tool error through the forward kinematics, a little joint speed, and a little set-point effort about $q^\star$:

$$
g(x,r) = 10\,\|p(q) - p^\star\|^2 + 0.01\,\|\dot q\|^2 + 0.1\,\|r - q^\star\|^2.
$$

It is written with JAX so it traces inside the compiled rollouts. The action that enters $g$ is the set-point $r$, the input of the inner loop.


In [ ]:
class ReachCost(CostFunction):
    def g(self, x, u, t=0.0, params=None):
        e = arm.forward_kinematics(x[:6]) - p_target
        return (
            10.0 * (e @ e)
            + 0.01 * (x[6:] @ x[6:])
            + 0.1 * ((u - q_target) @ (u - q_target))
        )

    def h(self, x, t=0.0, params=None):
        return 0.0

## 6. A stochastic planning problem

The RL problem is posed on that inner loop, not on the raw arm. `StochasticPlanningProblem` is the task: the plant is the impedance diagram, the cost is $g$, and two things are random.

1. The start, a Gaussian around a pose away from the target:
   $$x_0 \sim \mathcal N(x_{\mathrm{start}}, \Sigma).$$
2. The payload, an unknown mass on the last link that gravity compensation does not know. The dotted name `"sys.mass"` reaches the arm inside `impedance @ arm` (the plant subsystem is `sys`):
   $$m \sim \mathrm{Uniform}\big(m_{\mathrm{nom}},\; m_{\mathrm{nom}}\odot(1,1,1,1,1,3)\big).$$

The horizon is infinite; episodes last `EPISODE` seconds. The planner's action port is $r$, the inner loop's only input.


In [ ]:
masses = np.asarray(arm.params["mass"], dtype=float)
problem = StochasticPlanningProblem(
    inner,
    cost=ReachCost(),
    tf=np.inf,
    x0_distribution=Gaussian(
        inner.x0, np.concatenate([0.3 * np.ones(6), 0.1 * np.ones(6)])
    ),
    params_distribution={"sys.mass": Uniform(masses, masses * [1, 1, 1, 1, 1, 3.0])},
)
print(problem.sample_params(0))

## 7. A neural net, trained with RL

The third block is a network. What is learned is **only the outer set-point**. The impedance gains $K_p$, $K_d$ and the gravity term stay fixed. The network is a static map

$$
r = \pi_\theta\big(z(x)\big) \in \mathbb{R}^6,
$$

six joint references for the inner loop, not torques. The features $z$ are what a sensor would give — joint error, scaled rates, tool error — all order one:

$$
z(x) = \big(q - q^\star,\;\; 0.2\,\dot q,\;\; 3\big(p(q) - p^\star\big)\big).
$$

PPO adjusts $\theta$ to lower $J$ over random starts and payloads. A constant $r \equiv q^\star$ is the §4 baseline; the network can move $r$ when the payload pulls the tool off $p^\star$, which is why the **after** Bode is stiffer.

`ReinforcementLearningPlanner` runs many plants in parallel. The discrete discount $\gamma = 0.98$ is the per-step factor on the return; $g$ itself is not pre-discounted.


In [ ]:
planner = ReinforcementLearningPlanner(
    problem,
    dt=DT,
    features=lambda x: jnp.concatenate(
        [
            x[:6] - q_target,
            0.2 * x[6:],
            3.0 * (arm.forward_kinematics(x[:6]) - p_target),
        ]
    ),
    hidden=(64, 64),
    n_envs=32,
    n_steps=64,
    batch_size=256,
    learning_rate=3e-4,
    gamma=0.98,
    episode_length=EPISODE,
)
plan = planner.solve(timesteps=TRAINING_TIMESTEPS)
print(f"{plan.metadata.message} in {plan.metadata.solve_time_s:.0f} s")
planner.plot_learning_curve()

## 8. The network is a controller block

`get_controller()` returns a `NeuralPolicyController` — another `System`, same ports and equations as the impedance law. `MonteCarloEvaluator` scores it on the same draws the problem declared — the mean of $J$ over random starts and payloads.

The three blocks (network, impedance, arm) are one diagram. `@` cannot nest an already-closed inner loop, so the wires are written once. The tool force $f$ is a boundary input for the sensitivity Bode below. Two balls on the neural block: gold is the fixed task $p^\star$, green is the commanded inner-loop set-point $p(r)$. The crimson arrow on the arm is the perturbation $f$.


In [ ]:
rl_ctl = planner.get_controller()
rl_ctl.show_setpoint = True  # green ball: inner-loop command p(r)
rl_ctl.task_target = p_target  # gold ball: fixed task p*
print(
    "Monte Carlo:",
    MonteCarloEvaluator(
        problem, dt=DT, n_trials=64, episode_length=EPISODE, seed=1
    ).evaluate(rl_ctl),
)

loop = DiagramSystem()
loop.name = "Learned set-point law over joint impedance"
loop.add_subsystem(rl_ctl, "rl")
loop.add_subsystem(impedance, "ctl")
loop.add_subsystem(arm, "arm")
loop.connect("arm", "y", "rl", "x")
loop.connect("rl", "u", "ctl", "r")
loop.connect("arm", "y", "ctl", "y")
loop.connect("ctl", "u", "arm", "u")
loop.add_input_port("f", dim=3, nominal_value=np.zeros(3))
loop.connect("input", "f", "arm", "f")
loop.plot_diagram()

## 9. Watch the learned loop

Same start as the impedance-only run, now with the network in the loop. Compare the tool path $p$ and the mesh to §4 — then §10 asks the Bode question of this same diagram.


In [ ]:
loop.x0 = inner.x0
traj = loop.compute_trajectory(tf=EPISODE, dt=0.005, compile_backend="jax")
loop.plot_trajectory(traj, signals=((arm, "p"),))
loop.animate(traj, renderer="meshcat")

## 10. After RL: Bode

The destination: a frequency plot of a loop that contains a neural net. Same channel as §4 — vertical tool motion from a vertical tip force — linearized at the equilibrium the **learned** law holds. Compare the DC gain to the impedance-only Bode: a lower $|P_z/F_z|$ means the set-point law is pushing back against a steady load, like an integral term on $p$.


In [ ]:
x_eq = loop.find_equilibrium(x_target)
print("tool at equilibrium [m]:", np.round(arm.forward_kinematics(x_eq[:6]), 3))
loop.plot_bode(x_eq, of=("arm:p", 2), wrt=("f", 2), margins=False)

## 11. How far can it be pushed and still recover?

Bode says how the loop *responds*. It cannot say from how far it *recovers*: that question is nonlinear, and the linearization has already thrown the answer away. `region_of_attraction` asks it with three classical steps applied straight through the network — find the equilibrium (a root solve through $\pi_\theta$), linearize there (autodiff through the network, the impedance law and the arm), solve $A^\top P + PA = -Q$ for $V(x) = (x - \bar x)^\top P (x - \bar x)$ — and then one nonlinear sweep: the largest level set $\{V \le c\}$ on which $\dot V = \nabla V \cdot f$ stays negative, with $f$ the true loop, saturation and all. Every state inside returns to the equilibrium, and `verify()` draws states from that set and integrates them, so simulation gets its chance to break the claim.

Twelve states is also where a *sampled* sublevel search starts to run out: coverage thins fast with dimension, so the level comes out optimistic. The tool measures that itself, scoring two halves of its own samples against each other, and says `sample-limited` when they disagree. The plot is a slice of that set through the equilibrium — shoulder angle against shoulder rate, the other ten states pinned — with the dashed curve showing where $\dot V$ turns positive and stops the level. Read the number itself as an indication. The two-state version in [`demos/analysis/analysis_region_of_attraction.py`](../../demos/analysis/analysis_region_of_attraction.py) is where it is sharp — and where a stiffer gain turns out to prove *less* while its true basin stays the same size.

In [ ]:
certificate = loop.region_of_attraction(x_target)  # f = 0: nobody pushing
print(certificate)
print(certificate.verify())

certificate.plot(x_axis=1, y_axis=7)  # a slice: shoulder angle against its rate

## 12. Push the tool

The Bode channel in your hands. The learned loop's only boundary input is the tool force $f$. `game()` is a real-time loop: the 3D view is meshcat, held keys command $f$ in newtons (release returns to zero), and the neural set-point law and the impedance loop run as they did offline. The crimson arrow on the arm is that force; gold is the task $p^\star$, green is the commanded set-point $p(r)$. Keep the small pygame window focused for key events. Start at the learned equilibrium and push; ESC quits.

| keys | force |
| --- | --- |
| UP / DOWN | $f_x$ |
| RIGHT / LEFT | $f_y$ |
| W / S | $f_z$ |


In [ ]:
# Held keys command the tool force f [N]; release returns to 0
loop.inputs["f"].lower_bound = -20.0 * np.ones(3)
loop.inputs["f"].upper_bound = +20.0 * np.ones(3)
loop.game(renderer="meshcat", is_3d=True, x0=x_eq)

## 13. One framework

The same `System` contract carried every station on the path: six-axis rigid-body dynamics, a classical impedance loop, a neural net trained with PPO, a Bode plot of the closed loop, and a certificate of where that loop is guaranteed to recover.

| Station | What minilink used |
| --- | --- |
| UR5, $H(q)\ddot q + C\dot q + g(q) = \tau$ | catalog plant, `compute_trajectory`, `animate` |
| Joint impedance $\tau = K_p(r-q) - K_d\dot q + g(q)$ | `JointImpedance` `@` the arm |
| Network $r = \pi_\theta(x)$ | `ReinforcementLearningPlanner`, `NeuralPolicyController` |
| Closed loop $P_z/F_z$ | `plot_bode` on the wired diagram |
| Guarantee $\{V \le c\}$ | `region_of_attraction` on the same diagram |
| Same $f$, live | `game()` — push the tool |

A plant, a controller, a neural law, and a frequency plot are one kind of object. That is the bridge.

See also: the lab write-up [ur5_impedance_rl](../teaching/ur5_impedance_rl.ipynb), [showcase_jax](showcase_jax.ipynb), [11_reinforcement_learning](11_reinforcement_learning.ipynb).
